# CS331 — PrACo Test-1 | Official Adapter A + Frozen Qwen V7.6 (Google Colab)

**Đầu vào:** `CS331_PrACo_OFFICIAL_ADAPTER_A_FULL_FULL.zip` (34.510 cặp CountGD++ đã hoàn thành).

**Mục tiêu:** xây dựng dự đoán `Official adapter + V7.6`, **không chạy lại CountGD++**. Nếu có cache Qwen V7.6 text-only trước đây thì tái sử dụng các phán đoán hợp lệ; nếu không có thì Qwen phải suy luận mới cho các cặp baseline lớn hơn 0. Cặp baseline bằng 0 giữ nguyên 0 và không gọi Qwen.

**Quy tắc đã khóa:** chỉ trả số đếm 0 nếu baseline > 0 và cả hai lượt Qwen đều trả `NO`; với `YES`, `UNCERTAIN` hoặc đầu ra không hợp lệ thì giữ nguyên số đếm baseline. Prompt và revision cố định như Notebook B trước đây; không điều chỉnh dựa trên tập TEST.

**Quan trọng:** Đây là **PrACo Test-1 (Negative Label)**, chưa phải Test-2 (Mosaic). Chỉ tính chỉ số paired khi đã có đủ **34.510/34.510** dự đoán. Checkpoint mirror CountGD++ trong ZIP A chưa được đối chiếu bit với checkpoint tác giả; không gọi là tái lập weights chính thức hoàn toàn.

## Chuẩn bị Google Drive

Tạo thư mục `MyDrive/CS331_PrACo_B/` và đặt vào đó:

- **Bắt buộc:** `CS331_PrACo_OFFICIAL_ADAPTER_A_FULL_FULL.zip`.
- **Tùy chọn, giúp tiết kiệm inference:** `CS331_PrACo_V76_Precomputed_Qwen_Cache.zip` (nếu bạn có file cũ).
- **Nếu cần Qwen mới:** `hihihi5.zip` hoặc thư mục `FSC147_384_V2` chứa JPG của tập FSC-147. Có thể thay đường dẫn trong cell cấu hình.

**Lần 1:** `RUN_MODE='PLAN'` → xem có bao nhiêu cặp cần Qwen mới. **Lần 2:** `RUN_MODE='RUN'`, `MAX_NEW_PAIRS=3` → SMOKE 3 cặp. **Lần 3:** `MAX_NEW_PAIRS=0` → chạy tiếp FULL từ checkpoint trên Drive. Chọn GPU qua **Runtime → Change runtime type → T4 GPU** (hoặc L4/A100 nếu được cấp). Không yêu cầu chạy lại notebook A.

## 1. Gắn Google Drive và cấu hình chạy

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json, hashlib, time, shutil, zipfile, io, re
import numpy as np
import pandas as pd

drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/CS331_PrACo_B')
BASE_DIR.mkdir(parents=True, exist_ok=True)
OUT = BASE_DIR / 'B_RESULTS'
OUT.mkdir(parents=True, exist_ok=True)
TMP = Path('/content/praco_b_work')
TMP.mkdir(parents=True, exist_ok=True)

BASELINE_ZIP = BASE_DIR / 'CS331_PrACo_OFFICIAL_ADAPTER_A_FULL_FULL.zip'
OLD_QWEN_ZIP = BASE_DIR / 'CS331_PrACo_V76_Precomputed_Qwen_Cache.zip'  # Tùy chọn
IMAGE_SOURCE = BASE_DIR / 'hihihi5.zip'  # Hoặc BASE_DIR/'FSC147_384_V2' (thư mục)

RUN_MODE = 'PLAN'        # PLAN: không tải Qwen / không chạy GPU; RUN: thực thi
MAX_NEW_PAIRS = 3        # 3 = thử 3 cặp Qwen mới; 0 = chạy toàn bộ phần còn lại
SAVE_EVERY = 25          # Lưu CSV checkpoint sau mỗi 25 cặp Qwen mới

MODEL_ID = 'Qwen/Qwen3-VL-4B-Instruct'
QWEN_REVISION = 'ebb281ec70b05090aa6165b016eac8ec08e71b17'
EXPECTED_PRACO_COMMIT = '438a968aef9d31ded7519733bd84e89baae68896'
EXPECTED_ROSTER_SHA = 'b73d78d0e21d089cdd875913fe861929904bc7842e600e84ff5a562d439eae39'
EXPECTED_A_PROTOCOL_SHA = 'ab3a19afe6447047e71e4688fab39111b3c0ae90e02868a486bbecca341ba91c'
EXPECTED_A_MODEL_SHA = 'b89395d79711c9b9f13f3ef80cd2e528386fb1d70453f99443be4eccab33592f'

assert RUN_MODE in ('PLAN', 'RUN')
assert isinstance(MAX_NEW_PAIRS, int) and MAX_NEW_PAIRS >= 0
assert BASELINE_ZIP.is_file(), f'Thiếu ZIP A. Upload file vào: {BASELINE_ZIP}'
print('RUN_MODE:', RUN_MODE, '| MAX_NEW_PAIRS:', MAX_NEW_PAIRS)
print('A baseline:', BASELINE_ZIP)
print('Old Qwen cache:', 'Có' if OLD_QWEN_ZIP.exists() else 'Không có')
print('Ảnh FSC-147:', IMAGE_SOURCE if IMAGE_SOURCE.exists() else 'Chưa có; PLAN vẫn chạy được')

## 2. Đọc và kiểm tra đầy đủ 34.510 dự đoán từ ZIP A

In [ ]:
from zipfile import ZipFile

def sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()

def read_zipped_csv(z, filename, **kwargs):
    return pd.read_csv(io.BytesIO(z.read(filename)), **kwargs)

with ZipFile(BASELINE_ZIP) as z:
    bad = z.testzip()
    assert bad is None, f'CRC ZIP không đạt: {bad}'
    required = {'official_baseline_pairs.csv','official_test1_roster.csv',
                'Inference_Test1_CountGDPlusPlus_test.csv',
                'official_sources.json','official_baseline_protocol.json','run_state.json'}
    assert required.issubset(z.namelist()), f'Thiếu thành phần ZIP: {required-set(z.namelist())}'
    official = read_zipped_csv(z, 'official_baseline_pairs.csv', keep_default_na=False)
    roster = read_zipped_csv(z, 'official_test1_roster.csv', keep_default_na=False)
    matrix = read_zipped_csv(z, 'Inference_Test1_CountGDPlusPlus_test.csv', index_col=0)
    source_a = json.loads(z.read('official_sources.json'))
    protocol_a = json.loads(z.read('official_baseline_protocol.json'))
    run_state_a = json.loads(z.read('run_state.json'))

fields = ['pair_id','image_name','target','positive_class','exemplar_image']
assert len(roster)==len(official)==34510
assert len(set(roster.image_name))==1190 and len(set(roster.target))==29
assert official.pair_id.is_unique and roster.pair_id.is_unique
assert official.status.eq('ok').all()
assert set(official.pair_id)==set(roster.pair_id)
assert set(official.protocol_sha256)=={EXPECTED_A_PROTOCOL_SHA}
assert protocol_a['protocol_sha256']==EXPECTED_A_PROTOCOL_SHA
assert protocol_a['praco_commit']==source_a['adapter_git_sha']==EXPECTED_PRACO_COMMIT
assert source_a['checkpoint_sha256']==EXPECTED_A_MODEL_SHA
assert source_a['official_schedule_sha256']==EXPECTED_ROSTER_SHA
assert protocol_a['roster_sha256']==EXPECTED_ROSTER_SHA
assert sha256_bytes(roster[fields].to_csv(index=False).encode())==EXPECTED_ROSTER_SHA
assert set(pd.to_numeric(official.baseline_count,errors='coerce').dropna().index)==set(official.index)
assert np.isfinite(official.baseline_count).all() and (official.baseline_count>=0).all()
join = official[fields].merge(roster[fields],on='pair_id',suffixes=('_pred','_roster'),validate='one_to_one')
for k in fields[1:]:
    assert (join[f'{k}_pred']==join[f'{k}_roster']).all(), f'Không khớp roster: {k}'
check_matrix = official.pivot(index='image_name',columns='target',values='baseline_count')
check_matrix = check_matrix.reindex(index=matrix.index,columns=matrix.columns)
assert matrix.shape==(1190,29)
assert np.allclose(check_matrix.to_numpy(),matrix.to_numpy(),rtol=0,atol=1e-8)
assert run_state_a.get('completed')==34510

print('ZIP A: PASS | 34.510/34.510 | 1.190 ảnh x 29 lớp')
print('Protocol SHA:', EXPECTED_A_PROTOCOL_SHA)
print('CountGD++ checkpoint SHA:', EXPECTED_A_MODEL_SHA)
print('Số cặp baseline > 0:', int((official.baseline_count>0).sum()))
print('Nguồn checkpoint:', source_a['checkpoint_source'])

## 3. Nạp cache Qwen V7.6 nếu có và lập kế hoạch (KHÔNG dùng ground truth)

In [ ]:
FIRST_PROMPT = ('Target category: "{target}". Inspect the PHOTOGRAPH, not the question. '
 'Does the photograph contain at least one clearly identifiable, whole instance of EXACTLY this target category? '
 'Do not treat similar-looking categories, patterns, parts, text or pictures as real instances. '
 'If a genuine instance is clearly visible answer YES. If clearly absent answer NO. '
 'If too small, occluded, or visually ambiguous answer UNCERTAIN. '
 'Reply with exactly one word: YES, NO, or UNCERTAIN.')
SECOND_PROMPT = ('Target category: "{target}". Image one is the full photograph. '
 'Image two is a 2x2 montage of enlarged quadrants OF THE SAME PHOTO. '
 'Check carefully for even one true instance of the exact target in either view. '
 'Distinguish visually similar non-targets, and do not mistake a part or a picture for the object. '
 'YES if a real target is visible; NO only if the category is clearly absent in ALL views; '
 'otherwise UNCERTAIN. Reply with exactly one word: YES, NO, or UNCERTAIN.')

old = None
if OLD_QWEN_ZIP.exists():
    with ZipFile(OLD_QWEN_ZIP) as z:
        assert z.testzip() is None
        assert {'praco_gate_checkpoint.csv','frozen_protocol.json'}.issubset(z.namelist())
        old_df = read_zipped_csv(z, 'praco_gate_checkpoint.csv', keep_default_na=False)
        old_proto = json.loads(z.read('frozen_protocol.json'))
    assert len(old_df)==34510 and old_df.pair_id.is_unique
    assert set(old_df.pair_id)==set(official.pair_id)
    assert old_proto['qwen_model']==MODEL_ID
    assert old_proto['first_prompt']==FIRST_PROMPT
    assert old_proto['second_prompt']==SECOND_PROMPT
    assert set(old_df.model_revision.astype(str))=={QWEN_REVISION}
    assert old_df.status.eq('ok').all()
    old = old_df.set_index('pair_id')
    print('Old Qwen: PASS, 34.510 bản ghi; sẽ tái sử dụng verdict phù hợp.')
else:
    print('Không có old Qwen cache: cần suy luận Qwen mới cho các cặp baseline > 0.')

def semantic_replay(baseline, first, second):
    return 0.0 if float(baseline)>0 and first=='NO' and second=='NO' else float(baseline)

plan_rows=[]
# Index trực tiếp một lần cho tốc độ
base_index = official.set_index('pair_id')
for r in roster.itertuples(index=False):
    n=float(base_index.at[r.pair_id,'baseline_count'])
    f,s='NOT_RUN','NOT_RUN'
    reuse=False
    if old is not None and n>0:
        prev=old.loc[r.pair_id]
        old_n=float(prev['baseline_count'])
        f,s=str(prev['first_label']),str(prev['second_label'])
        reuse=(old_n>0 and f in ('YES','NO','UNCERTAIN')
               and (f!='NO' or s in ('YES','NO','UNCERTAIN')))
    mode=('ZERO_NO_QWEN' if n==0 else ('REUSE_QWEN' if reuse else 'NEW_QWEN_REQUIRED'))
    plan_rows.append(dict(pair_id=r.pair_id,image_name=r.image_name,target=r.target,
        positive_class=r.positive_class,baseline_count=n,replay_mode=mode,
        first_label=f if reuse else 'NOT_RUN',second_label=s if reuse else 'NOT_RUN',
        semantic_count=semantic_replay(n,f,s) if reuse else (0.0 if n==0 else np.nan)))

plan=pd.DataFrame(plan_rows)
assert len(plan)==34510 and plan.pair_id.is_unique
PLAN_CSV=OUT/'qwen_replay_plan_NO_GT.csv'
plan.to_csv(PLAN_CSV,index=False)
summary={k:int(v) for k,v in plan.replay_mode.value_counts().to_dict().items()}
(OUT/'plan_summary.json').write_text(json.dumps(summary,indent=2))
print('KẾ HOẠCH QWEN:',summary)
print('Đã lưu:',PLAN_CSV)

## 4. Chuẩn bị checkpoint trên Drive (không làm mất kết quả đã chạy)

In [ ]:
CKPT=OUT/'qwen_replay_checkpoint.csv'
CKPT_BACKUP=OUT/'qwen_replay_checkpoint_backup.csv'
PROTO_B={
    'stage':'PrACo Test1 official adapter + frozen V7.6',
    'official_checkpoint_sha':source_a['checkpoint_sha256'],
    'official_synth_json_sha':source_a['synthetic_json_sha256'],
    'official_protocol_sha':protocol_a['protocol_sha256'],
    'roster_sha':EXPECTED_ROSTER_SHA,
    'qwen_model':MODEL_ID,
    'qwen_revision':QWEN_REVISION,
    'prompt_sha256':sha256_bytes((FIRST_PROMPT+SECOND_PROMPT).encode()),
    'parse_policy':'exact single YES|NO|UNCERTAIN, else UNCERTAIN',
    'suppress_policy':'baseline>0 and first==NO and second==NO',
    'generation':'max_new_tokens=32, do_sample=False; original FP16 model',
}
PROTO_B_SHA=sha256_bytes(json.dumps(PROTO_B,sort_keys=True).encode())
(OUT/'paired_protocol.json').write_text(json.dumps({**PROTO_B,'protocol_sha256':PROTO_B_SHA},indent=2))

completed={}
for file in (CKPT_BACKUP,CKPT):
    if not file.exists():continue
    prior=pd.read_csv(file,keep_default_na=False)
    if prior.empty:continue
    assert prior.pair_id.is_unique and set(prior.pair_id).issubset(set(plan.pair_id))
    assert set(prior.protocol_sha256)=={PROTO_B_SHA},'Protocol B khác; không được trộn checkpoint!'
    assert prior.status.eq('ok').all()
    for row in prior.to_dict('records'):
        pid=row['pair_id']
        assert abs(float(row['baseline_count'])-float(base_index.at[pid,'baseline_count']))<1e-8
        assert np.isfinite(float(row['semantic_count'])) and float(row['semantic_count'])>=0
        if pid in completed:
            assert abs(float(completed[pid]['semantic_count'])-float(row['semantic_count']))<1e-8
        else:completed[pid]=row

for r in plan.itertuples(index=False):
    if r.pair_id in completed or r.replay_mode=='NEW_QWEN_REQUIRED':continue
    completed[r.pair_id]=dict(pair_id=r.pair_id,image_name=r.image_name,target=r.target,
        baseline_count=r.baseline_count,semantic_count=r.semantic_count,
        first_label=r.first_label,second_label=r.second_label,
        first_raw='',second_raw='',decision_source=r.replay_mode,
        status='ok',protocol_sha256=PROTO_B_SHA)

def save_checkpoint(stage='PARTIAL',make_zip=False):
    records=pd.DataFrame(list(completed.values())).sort_values('pair_id')
    tmp=TMP/'qwen_replay_checkpoint.csv'
    records.to_csv(tmp,index=False)
    staged=OUT/'qwen_replay_checkpoint.csv.tmp'
    shutil.copyfile(tmp,staged)
    if CKPT.exists():shutil.copy2(CKPT,CKPT_BACKUP)
    os.replace(staged,CKPT)
    state={'stage':stage,'completed':len(completed),'expected':34510,
           'remaining':34510-len(completed),'protocol_sha256':PROTO_B_SHA,
           'updated_at_utc':pd.Timestamp.utcnow().isoformat()}
    (OUT/'run_state.json').write_text(json.dumps(state,indent=2))
    if make_zip:
        path=OUT/f'CS331_PrACo_OFFICIAL_ADAPTER_B_{stage}.zip'
        with ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
            for p in [CKPT,PLAN_CSV,OUT/'paired_protocol.json',OUT/'run_state.json']:
                if p.exists():z.write(p,p.name)
        print('ZIP saved:',path)
    return state

print('Đã sẵn sàng / resume:',len(completed),'/34510')
print('Còn cần Qwen:',34510-len(completed))
if RUN_MODE=='RUN':
    save_checkpoint('PARTIAL',make_zip=True)
else:
    print('PLAN mode: chưa chạy GPU; chỉnh RUN_MODE=RUN để tiếp tục.')

## 5. Kiểm tra ảnh FSC-147 trước khi tải Qwen

In [ ]:
from pathlib import Path
from PIL import Image, ImageOps

pending = [r for r in plan.itertuples(index=False) if r.pair_id not in completed]
assert all(r.replay_mode=='NEW_QWEN_REQUIRED' for r in pending)
print('Chưa có verdict:',len(pending),'cặp /',len(set(r.image_name for r in pending)),'ảnh')
image_paths={}

if RUN_MODE=='RUN' and pending:
    expected_names=set(r.image_name for r in pending)
    local_images=TMP/'images'
    local_images.mkdir(parents=True,exist_ok=True)
    if IMAGE_SOURCE.is_dir():
        candidates={}
        for p in IMAGE_SOURCE.rglob('*'):
            if p.is_file() and p.suffix.lower() in ('.jpg','.jpeg') and p.name in expected_names:
                if p.name in candidates and p.read_bytes()!=candidates[p.name].read_bytes():
                    raise RuntimeError(f'Tên ảnh trùng nhưng nội dung khác: {p.name}')
                candidates[p.name]=p
        missing=expected_names-set(candidates)
        assert not missing,f'Thiếu {len(missing)} ảnh trong folder, ví dụ: {sorted(missing)[:10]}'
        image_paths=candidates
    elif IMAGE_SOURCE.is_file() and IMAGE_SOURCE.suffix.lower()=='.zip':
        with ZipFile(IMAGE_SOURCE) as z:
            assert z.testzip() is None,'ZIP ảnh bị hỏng'
            entries={}
            for item in z.infolist():
                name=Path(item.filename).name
                if not item.is_dir() and name in expected_names and name.lower().endswith(('.jpg','.jpeg')):
                    if name in entries and (entries[name].CRC != item.CRC or entries[name].file_size != item.file_size):
                        raise RuntimeError(f'Ảnh trùng tên nhưng khác nội dung trong ZIP: {name}')
                    entries[name]=item
            missing=expected_names-set(entries)
            assert not missing,f'Thiếu {len(missing)} ảnh trong ZIP, ví dụ: {sorted(missing)[:10]}'
            for name,item in entries.items():
                dest=local_images/name
                if not dest.exists() or dest.stat().st_size != item.file_size:
                    dest.write_bytes(z.read(item))
                image_paths[name]=dest
    else:
        raise FileNotFoundError('Chưa có ảnh FSC-147. Đặt hihihi5.zip hoặc folder FSC147_384_V2 vào MyDrive/CS331_PrACo_B rồi sửa IMAGE_SOURCE.')
    print('Ảnh hợp lệ:',len(image_paths),'| ví dụ:',list(image_paths.items())[:2])
elif RUN_MODE=='PLAN':
    print('PLAN: không cần ảnh ở bước này. Nếu NEW_QWEN_REQUIRED > 0, hãy chuẩn bị ảnh rồi RUN.')
else:
    print('Không cần suy luận mới: chỉ dùng ZERO/REUSE và checkpoint.')

## 6. Chỉ khi cần Qwen mới: cài môi trường và tải model đúng revision

In [ ]:
RUN_QWEN=(RUN_MODE=='RUN' and len(pending)>0)
if RUN_QWEN:
    import subprocess
    import torch
    assert torch.cuda.is_available(), 'Cần GPU Colab. Runtime → Change runtime type → T4 GPU hoặc GPU tốt hơn.'
    print('GPU:',torch.cuda.get_device_name(0),'| VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'transformers==4.57.1','huggingface-hub==0.36.2',
                    'accelerate>=1.8,<2','qwen-vl-utils==0.0.14'],check=True)
    import transformers
    assert transformers.__version__=='4.57.1', ('Transformers version không khớp, '
        'Runtime → Restart session rồi chạy lại toàn bộ cell.')
    from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
    from huggingface_hub import snapshot_download
    from qwen_vl_utils import process_vision_info
    proc_dir = snapshot_download(repo_id=MODEL_ID,revision=QWEN_REVISION,allow_patterns=[
        'config.json','processor_config.json','preprocessor_config.json',
        'video_preprocessor_config.json','tokenizer.json','tokenizer_config.json',
        'vocab.json','merges.txt','chat_template.json','chat_template.jinja',
        'special_tokens_map.json','added_tokens.json'])
    processor=AutoProcessor.from_pretrained(proc_dir,local_files_only=True)
    model=Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_ID, revision=QWEN_REVISION, dtype=torch.float16,
        device_map='auto', low_cpu_mem_usage=True).eval()
    print('Qwen loaded:',MODEL_ID,'at revision',QWEN_REVISION)
else:
    print('Không cần tải model Qwen (PLAN hoặc không có cặp cần inference mới).')

## 7. Qwen V7.6 frozen: parsing, montage và suy luận hai lượt

In [ ]:
if RUN_QWEN:
    import torch
    ONE_WORD_RX=re.compile(r'^(YES|NO|UNCERTAIN)[.!\s]*$',flags=re.I)
    def parse_answer(raw):
        match=ONE_WORD_RX.fullmatch(str(raw).strip())
        return match.group(1).upper() if match else 'UNCERTAIN'
    assert [parse_answer(x) for x in ['YES','No.','UNCERTAIN','YES NO','I think NO']]==[
        'YES','NO','UNCERTAIN','UNCERTAIN','UNCERTAIN']

    def quad_montage(image,side=336):
        W,H=image.size
        xs=[0,W//2,W];ys=[0,H//2,H]
        sheet=Image.new('RGB',(side*2,side*2),'white')
        for yi in range(2):
            for xi in range(2):
                patch=image.crop((xs[xi],ys[yi],xs[xi+1],ys[yi+1]))
                patch=ImageOps.contain(patch,(side,side))
                sheet.paste(patch,(xi*side+(side-patch.width)//2,
                                   yi*side+(side-patch.height)//2))
        return sheet

    @torch.inference_mode()
    def qwen_reply(image,target,step):
        image_msgs=[{'type':'image','image':image}]
        if step==2:
            image_msgs.append({'type':'image','image':quad_montage(image)})
        prompt=(FIRST_PROMPT if step==1 else SECOND_PROMPT).format(target=target)
        messages=[{'role':'user','content':image_msgs+[{'type':'text','text':prompt}]}]
        text_chat=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
        imgs,vids,kwargs=process_vision_info(messages,image_patch_size=16,
                              return_video_kwargs=True,return_video_metadata=True)
        videos_meta=None
        if vids is not None:
            vids,videos_meta=zip(*vids)
            vids=list(vids);videos_meta=list(videos_meta)
        model_in=processor(text=[text_chat],images=imgs,videos=vids,video_metadata=videos_meta,
                           padding=True,return_tensors='pt',do_resize=False,**kwargs)
        model_in=model_in.to(next(model.parameters()).device)
        generated=model.generate(**model_in,max_new_tokens=32,do_sample=False)
        raw=processor.batch_decode([o[len(i):] for i,o in zip(model_in.input_ids,generated)],
                   skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]
        del model_in,generated
        return raw,parse_answer(raw)
    print('Parser + montage + prompts: READY')
else:
    print('Bỏ qua định nghĩa Qwen: PLAN hoặc toàn bộ verdict đã có.')

## 8. Chạy SMOKE / FULL với checkpoint định kỳ trên Drive

In [ ]:
if RUN_MODE=='RUN' and pending:
    ran_this_session=0
    try:
        for row in pending:
            if MAX_NEW_PAIRS and ran_this_session >= MAX_NEW_PAIRS:
                print('Đã đạt giới hạn SMOKE của phiên:',MAX_NEW_PAIRS)
                break
            with Image.open(image_paths[row.image_name]) as f:
                img=f.convert('RGB')
            raw1,first=qwen_reply(img,row.target,step=1)
            raw2,second='','NOT_RUN'
            if first=='NO':
                raw2,second=qwen_reply(img,row.target,step=2)
            completed[row.pair_id]=dict(
                pair_id=row.pair_id,image_name=row.image_name,target=row.target,
                baseline_count=float(row.baseline_count),
                semantic_count=semantic_replay(row.baseline_count,first,second),
                first_label=first,second_label=second,
                first_raw=raw1,second_raw=raw2,
                decision_source='NEW_QWEN_INFERENCE',status='ok',
                protocol_sha256=PROTO_B_SHA)
            ran_this_session+=1
            if ran_this_session%SAVE_EVERY==0:
                save_checkpoint('PARTIAL')
                print('CHECKPOINT:',len(completed),'/34510; NEW this session:',ran_this_session)
    except BaseException:
        save_checkpoint('PARTIAL',make_zip=True)
        print('Đã lưu checkpoint trước khi kết thúc bất thường. Có thể chạy lại để resume.')
        raise
    finally:
        save_checkpoint('PARTIAL',make_zip=True)
    print('Đã chạy Qwen mới trong phiên:',ran_this_session)
elif RUN_MODE=='PLAN':
    print('PLAN: Chưa chạy Qwen. Đổi RUN_MODE=RUN, MAX_NEW_PAIRS=3 để thử.')
else:
    print('Không còn cặp nào cần Qwen mới.')
print('TỔNG HOÀN THÀNH:',len(completed),'/34510')

## 9. Niêm phong predictions khi đủ 34.510 và đánh giá theo PrACo Test-1

In [ ]:
def digest_file(path,chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(chunk),b''):
            h.update(b)
    return h.hexdigest()

def paired_test1(df,image_classes,gt,expected_images,targets):
    assert len(df)==len(expected_images)*len(targets)
    scores=[]
    for col in ('baseline_count','semantic_count'):
        mat=df.pivot(index='image_name',columns='target',values=col)
        assert not mat.isna().any().any()
        mat=mat.reindex(index=expected_images,columns=targets)
        assert not mat.isna().any().any()
        pos,neg=[],[]
        for im in mat.index:
            g=float(gt[im]);t=image_classes[im]
            assert g>0 and t in mat.columns
            p=float(mat.at[im,t]); n=float(mat.loc[im,mat.columns!=t].mean())
            pos.append((p,g));neg.append(n)
        g=np.array([x[1] for x in pos]);p=np.array([x[0] for x in pos]);n=np.array(neg)
        scores.append(dict(Model=col,n_images=len(g),AvgNP=np.mean(p/g),
            AvgNMN=np.mean(n/g),PCCN=100*np.mean(np.abs(p-g)<np.abs(n-g)),
            MAE=np.mean(np.abs(p-g)),RMSE=np.sqrt(np.mean((p-g)**2)),
            mismatch_nonzero_rate=float((mat.to_numpy()>0).sum()-(p>0).sum())/(len(g)*(len(targets)-1)),
            positive_zero=int((p==0).sum())))
    return pd.DataFrame(scores)

if RUN_MODE=='RUN' and len(completed)==34510:
    result=pd.DataFrame(list(completed.values())).sort_values('pair_id').reset_index(drop=True)
    assert result.pair_id.is_unique and result.status.eq('ok').all()
    assert set(result.pair_id)==set(official.pair_id)
    assert set(result.protocol_sha256)=={PROTO_B_SHA}
    for col in ('baseline_count','semantic_count'):
        assert np.isfinite(pd.to_numeric(result[col])).all()
        assert (pd.to_numeric(result[col])>=0).all()
    assert np.allclose(result.set_index('pair_id').loc[base_index.index,'baseline_count'].to_numpy(),
                       base_index.baseline_count.to_numpy(),rtol=0,atol=1e-8)
    SEALED=OUT/'praco_official_plus_v76_SEALED.csv'
    result.to_csv(SEALED,index=False)
    pred_sha=digest_file(SEALED)
    (OUT/'prediction_sha256.txt').write_text(f'{pred_sha}  {SEALED.name}\n')
    print('PREDICTIONS SEALED:',pred_sha)
    save_checkpoint('FULL')

    # Tải ground truth CHỈ SAU khi predictions đã niêm phong
    GT_FILE=OUT/'gt_cnts_FSC147_official.json'
    ROOT='https://raw.githubusercontent.com/ciampluca/PrACo/'+EXPECTED_PRACO_COMMIT+'/data/'
    if not GT_FILE.exists():
        import urllib.request
        urllib.request.urlretrieve(ROOT+'gt_cnts_FSC147.json',GT_FILE)
    gt=json.loads(GT_FILE.read_text())
    image_classes=dict(zip(official.image_name,official.positive_class))
    images=roster.image_name.drop_duplicates().tolist()
    targets=roster.target.drop_duplicates().tolist()
    assert len(images)==1190 and len(targets)==29
    metrics=paired_test1(result,image_classes,gt,images,targets)
    metrics.to_csv(OUT/'PrACo_OfficialAdapter_Paired_Test1_metrics.csv',index=False)
    print('PRACO TEST-1 FULL METRICS:')
    print(metrics.to_string(index=False,float_format=lambda x:f'{x:.6f}'))
    FINAL_ZIP=OUT/'CS331_PrACo_OFFICIAL_ADAPTER_B_FULL.zip'
    with ZipFile(FINAL_ZIP,'w',zipfile.ZIP_DEFLATED) as z:
        for fn in [SEALED,OUT/'prediction_sha256.txt',OUT/'PrACo_OfficialAdapter_Paired_Test1_metrics.csv',
                   CKPT,PLAN_CSV,OUT/'paired_protocol.json',OUT/'run_state.json']:
            z.write(fn,fn.name)
    print('DONE:',FINAL_ZIP)
    print('LƯU Ý: Test-1 class-mismatch != verified-absent FPR; Test-2 Mosaic chưa chạy.')
else:
    print('PARTIAL / PLAN:',len(completed),'/34510 — CHƯA TÍNH FULL METRICS.')
    if RUN_MODE=='RUN':
        print('Muốn chạy tiếp: đổi MAX_NEW_PAIRS=0 rồi Runtime → Run all; checkpoint vẫn ở Drive.')

## 10. Tệp kết quả và lưu ý phục hồi

Tất cả lưu trên Google Drive: `MyDrive/CS331_PrACo_B/B_RESULTS/`.

- `qwen_replay_plan_NO_GT.csv`: kế hoạch ban đầu, không dùng ground truth.
- `qwen_replay_checkpoint.csv` và `qwen_replay_checkpoint_backup.csv`: checkpoint có thể resume.
- `CS331_PrACo_OFFICIAL_ADAPTER_B_PARTIAL.zip`: kết quả dở dang sau từng phiên.
- `praco_official_plus_v76_SEALED.csv` và `PrACo_OfficialAdapter_Paired_Test1_metrics.csv`: **chỉ tạo khi đủ 34.510 cặp**.
- `CS331_PrACo_OFFICIAL_ADAPTER_B_FULL.zip`: gói kết quả cuối cùng.

**Không đổi** prompt, model revision, precision, cách tạo montage hoặc protocol sau khi chạy. Nếu Colab disconnect: mở notebook, kết nối GPU, **chạy lại từ đầu các cell**; checkpoint trên Drive được kiểm tra SHA và tự dùng lại. Không xóa `B_RESULTS`.

**GPU / an toàn:** Google Colab có thể thu hồi GPU hoặc ngắt runtime; GPU T4 có thể hết VRAM khi tải Qwen FP16. Nếu OOM, chuyển runtime sang GPU nhiều VRAM hơn rồi resume; **không tự quantize, resize, đổi model** và gộp kết quả dưới cùng protocol. Không có cache Qwen cũ thì số lượt mới nhiều hơn; Colab có thể cần nhiều phiên.